## **Aim**
To implement a program that identifies suspicious IP addresses from a network security log using predefined threat indicators.

## **Algorithm**
**Step 1:** Import `re`, `collections.Counter`, `json`, and `datetime` libraries.

**Step 2:** Create a simulated network security log (JSON) with fields: timestamp, source_ip, destination_ip, protocol, port, bytes, action (allow/deny).

**Step 3:** Define threat indicators:
   - Known malicious IP ranges/CIDR
   - Tor exit nodes
   - VPN/proxy IPs
   - High-risk countries
   - IPs with high failed connection counts
   - IPs scanning multiple ports

**Step 4:** Parse the log and extract source IPs with their activity patterns.

**Step 5:** Score each IP based on threat indicators.

**Step 6:** Generate a report of suspicious IPs with risk scores and evidence.

In [1]:
import re
import json
import shutil
from collections import Counter, defaultdict
from datetime import datetime, timedelta

MALICIOUS_IPS = {
    "192.168.100.50": "Known C2 server",
    "10.0.0.200": "Known malware host",
}

TOR_EXIT_NODES = {
    "203.0.113.45", "198.51.100.23", "192.0.2.100",
}

VPN_PROXY_IPS = {
    "198.51.100.23", "192.0.2.50", "203.0.113.100",
}

HIGH_RISK_COUNTRIES = {
    "CN", "RU", "KP", "IR", "SY",
}

IP_COUNTRY = {
    "203.0.113.45": "CN",
    "198.51.100.23": "RU",
    "192.0.2.100": "DE",
    "192.168.100.50": "XX",
    "10.0.0.100": "INTERNAL",
    "172.16.0.50": "INTERNAL",
    "192.168.1.50": "INTERNAL",
    "192.168.1.200": "INTERNAL",
    "10.0.0.200": "INTERNAL",
    "172.16.0.10": "INTERNAL",
    "192.168.1.10": "INTERNAL",
    "8.8.8.8": "US",
}

def create_sample_network_log(log_file):
    now = datetime.now()
    base = now - timedelta(hours=24)
    
    events = []
    
    # Normal traffic
    for i in range(50):
        events.append({
            "timestamp": (base + timedelta(minutes=i*20)).isoformat(),
            "source_ip": "192.168.1.50",
            "dest_ip": "8.8.8.8",
            "protocol": "UDP",
            "port": 53,
            "bytes": 100,
            "action": "ALLOW"
        })
    
    for i in range(30):
        events.append({
            "timestamp": (base + timedelta(minutes=i*30)).isoformat(),
            "source_ip": "192.168.1.200",
            "dest_ip": "192.168.1.10",
            "protocol": "TCP",
            "port": 443,
            "bytes": 5000,
            "action": "ALLOW"
        })
    
    # C2 traffic
    for i in range(45):
        events.append({
            "timestamp": (base + timedelta(minutes=i*30)).isoformat(),
            "source_ip": "192.168.100.50",
            "dest_ip": f"10.0.0.{i%20+1}",
            "protocol": "TCP",
            "port": 443 if i % 2 == 0 else 80,
            "bytes": 10000 if i % 3 == 0 else 500,
            "action": "ALLOW" if i % 4 != 0 else "DENY"
        })
    
    # Internal scanner
    for i in range(30):
        events.append({
            "timestamp": (base + timedelta(minutes=i*40)).isoformat(),
            "source_ip": "10.0.0.100",
            "dest_ip": f"192.168.1.{i%25}",
            "protocol": "TCP",
            "port": 22 + (i % 20),
            "bytes": 100,
            "action": "DENY" if i % 2 == 0 else "ALLOW"
        })
    
    # Brute force
    for i in range(25):
        events.append({
            "timestamp": (base + timedelta(minutes=i*10)).isoformat(),
            "source_ip": "172.16.0.50",
            "dest_ip": "192.168.1.10",
            "protocol": "TCP",
            "port": 22,
            "bytes": 200,
            "action": "DENY" if i % 3 != 0 else "ALLOW"
        })
    
    # Tor exit node
    for i in range(15):
        events.append({
            "timestamp": (base + timedelta(hours=i)).isoformat(),
            "source_ip": "203.0.113.45",
            "dest_ip": "192.168.1.10",
            "protocol": "TCP",
            "port": 443,
            "bytes": 300,
            "action": "ALLOW"
        })
    
    # VPN/Proxy
    for i in range(12):
        events.append({
            "timestamp": (base + timedelta(hours=i*2)).isoformat(),
            "source_ip": "198.51.100.23",
            "dest_ip": "192.168.1.10",
            "protocol": "TCP",
            "port": 80,
            "bytes": 200,
            "action": "ALLOW"
        })
    
    with open(log_file, "w") as f:
        json.dump(events, f, indent=2)

def analyze_network_log(log_file):
    with open(log_file, "r") as f:
        events = json.load(f)
    
    # Group by source IP
    ip_stats = defaultdict(lambda: {
        "connections": 0,
        "destinations": set(),
        "ports": set(),
        "denied": 0,
        "allowed": 0,
        "bytes_total": 0,
        "protocols": set(),
    })
    
    for e in events:
        ip = e["source_ip"]
        stats = ip_stats[ip]
        stats["connections"] += 1
        stats["destinations"].add(e["dest_ip"])
        stats["ports"].add(e["port"])
        stats["protocols"].add(e["protocol"])
        stats["bytes_total"] += e["bytes"]
        if e["action"] == "DENY":
            stats["denied"] += 1
        else:
            stats["allowed"] += 1
    
    # Score each IP
    scored_ips = []
    
    for ip, stats in ip_stats.items():
        score = 0
        indicators = []
        
        # Known malicious IP
        if ip in MALICIOUS_IPS:
            score += 50
            indicators.append(f"Known malicious IP: {MALICIOUS_IPS[ip]}")
        
        # Tor exit node
        if ip in TOR_EXIT_NODES:
            score += 25
            indicators.append("Tor exit node")
        
        # VPN/Proxy
        if ip in VPN_PROXY_IPS:
            score += 15
            indicators.append("VPN/Proxy IP")
        
        # High-risk country
        country = IP_COUNTRY.get(ip, "XX")
        if country in HIGH_RISK_COUNTRIES:
            score += 20
            indicators.append(f"High-risk country: {country}")
        
        # High denied connection rate
        if stats["connections"] > 0:
            deny_rate = stats["denied"] / stats["connections"]
            if deny_rate > 0.5:
                score += 20
                indicators.append(f"High denied connection rate ({deny_rate:.0%})")
        
        # High port diversity (scanning)
        if len(stats["ports"]) >= 10:
            score += 15
            indicators.append(f"High port diversity ({len(stats['ports'])} unique ports)")
        
        # High destination diversity
        if len(stats["destinations"]) >= 15:
            score += 10
            indicators.append(f"High destination diversity ({len(stats['destinations'])} unique destinations)")
        
        # Exfiltration pattern (large bytes, few connections)
        if stats["bytes_total"] > 100000 and stats["connections"] > 10:
            score += 15
            indicators.append("Exfiltration pattern detected")
        
        # Brute force pattern (same port, many denied)
        if stats["denied"] >= 10 and len(stats["ports"]) <= 3:
            score += 15
            indicators.append("Brute force pattern detected")
        
        # Internal reconnaissance
        if ip.startswith("10.") or ip.startswith("172.16.") or ip.startswith("192.168."):
            if len(stats["destinations"]) >= 10 and stats["denied"] >= 10:
                score += 10
                indicators.append("Internal reconnaissance")
        
        # Determine risk
        if score >= 70:
            risk = "CRITICAL"
        elif score >= 50:
            risk = "HIGH"
        elif score >= 25:
            risk = "MEDIUM"
        elif score > 0:
            risk = "LOW"
        else:
            risk = "SAFE"
        
        if score > 0:
            scored_ips.append({
                "ip": ip,
                "score": score,
                "risk": risk,
                "indicators": indicators,
                "stats": stats,
                "country": country
            })
    
    scored_ips.sort(key=lambda x: -x["score"])
    return scored_ips, ip_stats

def main():
    log_file = "network_security_log.json"
    create_sample_network_log(log_file)
    
    print("Analyzing network security log for suspicious IPs...")
    scored_ips, ip_stats = analyze_network_log(log_file)
    
    print(f"\n{'='*60}")
    print(f"SUSPICIOUS IP ADDRESS REPORT")
    print(f"{'='*60}")
    print(f"Total unique source IPs: {len(ip_stats)}")
    total_entries = sum(s["connections"] for s in ip_stats.values())
    print(f"Total log entries: {total_entries}")
    
    print(f"\n--- TOP SUSPICIOUS IPs ---")
    
    for i, ip_info in enumerate(scored_ips, 1):
        s = ip_info["stats"]
        print(f"\n{i}. [{ip_info['risk']}] {ip_info['ip']} (Score: {ip_info['score']})")
        print(f"    Country: {ip_info['country']}")
        print(f"    Connections: {s['connections']}")
        print(f"    Unique Destinations: {len(s['destinations'])}")
        print(f"    Unique Ports: {len(s['ports'])}")
        print(f"    Denied Connections: {s['denied']}")
        print(f"    Indicators:")
        for ind in ip_info["indicators"]:
            print(f"      - {ind}")
    
    # Summary
    from collections import Counter
    risk_counts = Counter(ip["risk"] for ip in scored_ips)
    safe_count = len(ip_stats) - len(scored_ips)
    print(f"\n--- SUMMARY ---")
    for risk in ["CRITICAL", "HIGH", "MEDIUM", "LOW", "SAFE"]:
        count = risk_counts.get(risk, 0)
        if risk == "SAFE":
            count = safe_count
        if count > 0:
            print(f"  {risk}: {count}")

if __name__ == "__main__":
    main()

Analyzing network security log for suspicious IPs...

SUSPICIOUS IP ADDRESS REPORT
Total unique source IPs: 7
Total log entries: 207

--- TOP SUSPICIOUS IPs ---

1. [CRITICAL] 192.168.100.50 (Score: 100)
    Country: XX
    Connections: 45
    Unique Destinations: 20
    Unique Ports: 2
    Denied Connections: 12
    Indicators:
      - Known malicious IP: Known C2 server
      - High destination diversity (20 unique destinations)
      - Exfiltration pattern detected
      - Brute force pattern detected
      - Internal reconnaissance

2. [HIGH] 198.51.100.23 (Score: 60)
    Country: RU
    Connections: 12
    Unique Destinations: 1
    Unique Ports: 1
    Denied Connections: 0
    Indicators:
      - Tor exit node
      - VPN/Proxy IP
      - High-risk country: RU

3. [MEDIUM] 203.0.113.45 (Score: 45)
    Country: CN
    Connections: 15
    Unique Destinations: 1
    Unique Ports: 1
    Denied Connections: 0
    Indicators:
      - Tor exit node
      - High-risk country: CN

4. [MED

## **Result**
This the program successfully identifies suspicious IP addresses from a network security log using predefined threat indicators.